In [1]:


import numpy as np 
import pandas as pd 


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/claudiacornacchia/tweets/tweets.csv
/kaggle/input/datasets/claudiacornacchia/embedding/base_map.csv
/kaggle/input/datasets/claudiacornacchia/embedding/bertopic_data.json
/kaggle/input/datasets/claudiacornacchia/embedding/umap_mapper.joblib
/kaggle/input/datasets/claudiacornacchia/embedding/embeddings_1024d.pt


# Personal setting

In [2]:
BASE_INPUT_PATH = "/kaggle/input/datasets/claudiacornacchia/"
EMBEDDING_DIR = "embedding" 
TWEETS_DIR = "tweets"
BASE_WORKING_PATH = "/kaggle/working/"
PORT= 5000 

## Install dependencies 

In [3]:
!pip install umap-learn spacy nltk
!python -m spacy download en_core_web_sm
!pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.2 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
!pip install flask flask-cors pyngrok


In [5]:
!pip install bertopic

## Imports

In [6]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
import umap
import spacy
from nltk.corpus import wordnet
import nltk
from tqdm import tqdm
import os
import joblib 
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
import threading
import json
import requests

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

nlp = spacy.load("en_core_web_sm")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

2026-07-23 15:00:46.392322: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784818846.415907    1321 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784818846.423497    1321 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784818846.443818    1321 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784818846.443837    1321 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784818846.443839    1321 computation_placer.cc:177] computation placer alr

Using device: cuda



In [7]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

conf.get_default().ngrok_path = "/usr/bin/ngrok"


user_secrets = UserSecretsClient()
my_secret_token = user_secrets.get_secret("NGROK_TOKEN")

In [8]:
global_vectorizer = None
global_feature_names = None

## Model initialization

In [9]:
# Load Embedding Model (Qwen 0.6B)
emb_model_id = "Qwen/Qwen3-Embedding-0.6B"
print(f"Loading embedding model {emb_model_id}...")
emb_tokenizer = AutoTokenizer.from_pretrained(emb_model_id, trust_remote_code=True)
emb_tokenizer.padding_side = "right"
emb_model = AutoModel.from_pretrained(emb_model_id, trust_remote_code=True).to(device)
emb_model.eval()

# Load Generative Model (Gemma 4 E2B Instruct)
gen_model_id = "google/gemma-4-E2B-it"
print(f"Loading generative model {gen_model_id}...")

gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_id)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_id, 
    torch_dtype=torch.bfloat16,
    device_map="cuda:0"
)



Loading embedding model Qwen/Qwen3-Embedding-0.6B...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading generative model google/gemma-4-E2B-it...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

## Embedding functions

In [10]:
def get_embedding(text):
    """Calculates the embedding for a single text string."""
    inputs = emb_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = emb_model(**inputs)
        attention_mask = inputs['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embedding = sum_embeddings / sum_mask
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding

def batch_get_embeddings(texts, batch_size=32):
    """Calculates embeddings for a list of texts efficiently in batches."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Calculating Batch Embeddings"):
        batch_texts = texts[i : i + batch_size]
        inputs = emb_tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = emb_model(**inputs)
            attention_mask = inputs['attention_mask']
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            embeddings = sum_embeddings / sum_mask
            embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

## Perturbation logic

In [11]:

def get_wordnet_pos_from_spacy(spacy_pos):
    """Maps spaCy POS tags to WordNet formats."""
    if spacy_pos == 'ADJ': return wordnet.ADJ
    elif spacy_pos == 'VERB': return wordnet.VERB
    elif spacy_pos == 'NOUN': return wordnet.NOUN
    elif spacy_pos == 'ADV': return wordnet.ADV
    return None
def perturb_dict(text, mode="synonym", target_word=None, replacement_word=None):
    doc = nlp(text)
    perturbed_text = ""
    display_text = "" 
    changed = False
    target_found = False

    if mode == "custom":
        if not target_word:
            raise ValueError("For the custom replace you should select a word in the tweet first.")
        if not replacement_word or not replacement_word.strip():
            raise ValueError("Write the new word.")

    for token in doc:
        if target_word:
            if token.text.lower() == target_word.lower():
                target_found = True

                if mode == "custom":
                    perturbed_text += replacement_word + token.whitespace_
                    display_text += replacement_word + token.whitespace_
                    changed = True
               
                elif mode == "remove":
                    perturbed_text += token.whitespace_
                    display_text += token.whitespace_
                    changed = True
                else:
                    replacement = get_single_word_replacement(token, mode=mode)
                    if replacement:
                        perturbed_text += replacement + token.whitespace_
                        display_text += f"<strong>{replacement}</strong>{token.whitespace_}"
                        changed = True
                    else:
                        
                        perturbed_text += token.text_with_ws
                        display_text += token.text_with_ws
            else:
            
                perturbed_text += token.text_with_ws
                display_text += token.text_with_ws

    
        else:
            
            if changed or not token.is_alpha:
                perturbed_text += token.text_with_ws
                display_text += token.text_with_ws
                continue
            if mode == "remove":
                perturbed_text += token.whitespace_
                display_text += token.whitespace_
                changed = True
            else:
                replacement = get_single_word_replacement(token, mode=mode)
                if replacement:
                    perturbed_text += replacement + token.whitespace_
                    display_text += replacement + token.whitespace_
                    changed = True
                else:
                    perturbed_text += token.text_with_ws
                    display_text += token.text_with_ws

  
    if target_word:
        if not target_found:
            raise ValueError(f"The word '{target_word}' is not present in this tweet.")
        if target_found and not changed:
            raise ValueError(f"No {mode} found in WordNet for the word '{target_word}'.")
    else:
        if not changed:
            raise ValueError(f"No word in this tweet has a {mode} in the dictionary.")
    
    return perturbed_text, display_text

def get_single_word_replacement(spacy_token, mode="synonym"):
    """
    Cerca un sinonimo/contrario. Prima prova WordNet, se fallisce (per slang o mancanza di copertura)
    interroga l'API di Datamuse per trovare l'alternativa.
    """
    
    wn_tag = get_wordnet_pos_from_spacy(spacy_token.pos_)
    replacements = set()
    
    if wn_tag:
        synsets = wordnet.synsets(spacy_token.lemma_, pos=wn_tag)
        for syn in synsets:
            for lemma in syn.lemmas():
                if mode == "synonym" and lemma.name().lower() != spacy_token.lemma_.lower():
                    replacements.add(lemma.name().replace('_', ' '))
                elif mode == "antonym" and lemma.antonyms():
                    replacements.add(lemma.antonyms()[0].name().replace('_', ' '))


    if not replacements:
        try:
            word = spacy_token.text.lower()
            if mode == "synonym":
                # ml = meaning like
                url = f"https://api.datamuse.com/words?ml={word}&max=5"
            else:
                # rel_ant = antonyms
                url = f"https://api.datamuse.com/words?rel_ant={word}&max=5"
                
            response = requests.get(url, timeout=2).json()
            for item in response:
                if " " not in item['word']: 
                    replacements.add(item['word'])
        except Exception:
            pass 


    if replacements:
        new_word = random.choice(list(replacements))
    
        if spacy_token.is_title: 
            new_word = new_word.title()
        elif spacy_token.is_upper: 
            new_word = new_word.upper()
        return new_word
        
    return None

import random

def perturb_noise(text):
    """
    3. Noise Perturbation: Inserts random noise words into the tweet.
    Updated with literature-backed noise categories (Disfluencies, OOV, Slang).
    """
    doc = nlp(text)
    if len(doc) == 0: return text, text
    
    
    academic_noise_words = [
        # Disfluencies (CheckList)
        "uh", "um", "uhm", "err", "ah", "like", "literally", "basically", "actually", 
        
        # Synthetic / Keyboard mash
        "blah", "xyz", "asdf", "qwerty", "foo", "bar",
        
        # Twitter/Social Media structural noise
        "lol", "lmao", "smh", "tbh", "btw", "ikr", "omg"
    ]
    
    tokens_with_ws = [token.text_with_ws for token in doc]
    display_tokens = [token.text_with_ws for token in doc]
    

    num_noise = random.randint(1, 3) 
    
    for _ in range(num_noise):
        insert_idx = random.randint(0, len(tokens_with_ws))
        noise = random.choice(academic_noise_words)
        
        
        tokens_with_ws.insert(insert_idx, noise + " ")
        
        
        display_tokens.insert(insert_idx, f"<span class='perturbation-diff'>{noise}</span> ")
        
    new_text = "".join(tokens_with_ws)
    display_text = "".join(display_tokens)
    
    return new_text, display_text

def perturb_llm(text, mode="paraphrase"):
    """
    4 & 5. LLM Perturbation: Rephrases or flips sentiment completely.
    """
    if mode == "paraphrase":
        instruction = "Rewrite the following airline review tweet using completely different words, but keep the exact same meaning and sentiment."
    elif mode == "opposite":
        instruction = "Rewrite the following airline review tweet to have the exact opposite sentiment. If it complains, make it praise the airline. If it praises, make it complain."
        
    messages = [
        {
            "role": "system", 
            "content": "You are a helpful assistant specialized in rewriting text. Output ONLY the rewritten text, without explanations, conversational filler, markdown blocks, or quotes."
        },
        {
            "role": "user", 
            "content": f"{instruction}\n\nTweet: {text}"
        }
    ]
    
    text_input = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = gen_tokenizer([text_input], return_tensors="pt").to(gen_model.device)    
    with torch.no_grad():
        generated_ids = gen_model.generate(
            **model_inputs, 
            max_new_tokens=100, 
            temperature=0.7,
            top_p=0.95,         
            top_k=64,           
            do_sample=True
        )

    
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    torch.cuda.empty_cache() 
    return gen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


def perturb_typo(text):
    """
    1. Typo Perturbation: Inverte due lettere preservando spazi e formattazione originali.
    """
    doc = nlp(text)
    tokens = [t for t in doc]
    
    candidates = [i for i, t in enumerate(tokens) if len(t.text) > 4 and t.is_alpha]
    
    if not candidates:
        return text, text 
        
    target_idx = random.choice(candidates)
    word = tokens[target_idx].text
    
    swap_idx = random.randint(1, len(word) - 3)
    new_word = word[:swap_idx] + word[swap_idx+1] + word[swap_idx] + word[swap_idx+2:]
    
    
    if tokens[target_idx].is_title: new_word = new_word.title()
    elif tokens[target_idx].is_upper: new_word = new_word.upper()
        
    new_text = ""
    display_text = ""
    
    for i, t in enumerate(tokens):
        if i == target_idx:
            new_text += new_word + t.whitespace_
            
            display_text += f"<strong style='color: #f97316; text-decoration: underline;'>{new_word}</strong>{t.whitespace_}"
        else:
            new_text += t.text_with_ws
            display_text += t.text_with_ws
            
    return new_text, display_text

def perturb_shuffle(text):
    """
    2. Syntax Attack: Mescola solo le parole vere, ignorando punteggiatura e preservando spazi.
    """
    doc = nlp(text)
    word_tokens = [t for t in doc if t.is_alpha]
    
    if len(word_tokens) <= 2:
        return text, text
        
    shuffled_texts = [t.text for t in word_tokens]
    random.shuffle(shuffled_texts)
    
    new_text = ""
    display_text = ""
    word_idx = 0
    
    for t in doc:
        if t.is_alpha:
            new_word = shuffled_texts[word_idx]
            new_text += new_word + t.whitespace_
            display_text += f"<strong style='color: #8b5cf6;'>{new_word}</strong>{t.whitespace_}"
            word_idx += 1
        else:
            new_text += t.text_with_ws
            display_text += t.text_with_ws
            
    return new_text, display_text

def perturb_leetspeak(text):
    """
    3. Leetspeak Attack: Sostituisce i caratteri con numeri visivamente simili.
    """
    doc = nlp(text)
    leet_map = {'a': '4', 'A': '4', 'e': '3', 'E': '3', 'i': '1', 'I': '1', 'o': '0', 'O': '0', 's': '$', 'S': '$', 't': '7', 'T': '7'}
    
    new_text = ""
    display_text = ""
    changed_any = False
    
    for t in doc:
        if t.is_alpha and len(t.text) > 3:
            perturbed_word = ""
            changed_word = False
            for char in t.text:
                if char in leet_map and random.random() > 0.4:
                    perturbed_word += leet_map[char]
                    changed_word = True
                    changed_any = True
                else:
                    perturbed_word += char
                    
            new_text += perturbed_word + t.whitespace_
            if changed_word:
                display_text += f"<strong style='color: #10b981;'>{perturbed_word}</strong>{t.whitespace_}"
            else:
                display_text += perturbed_word + t.whitespace_
        else:
            new_text += t.text_with_ws
            display_text += t.text_with_ws
            
    if not changed_any: return text, text
    return new_text, display_text



## TF-IDF

In [12]:
def setup_tfidf(testi_globali):
    global global_vectorizer, global_feature_names
    print("Addestramento TF-IDF globale in corso...")
    
    global_vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)
    global_vectorizer.fit(testi_globali)
    global_feature_names = global_vectorizer.get_feature_names_out()
    print("TF-IDF pronto.")

## Bertopic

In [13]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS 

global_topics_data = []

def initialize_bertopic(tweets_texts, precalculated_embeddings):
    global global_topics_data
    print("Initializing BERTopic (with Custom Stop-Words)...")

   
    base_stopwords = list(ENGLISH_STOP_WORDS)
    
    airline_stopwords = [
        "united", "usairways", "americanair", "southwestair", "jetblue",
        "jetblues", "virginamerica", "virgin", "southwest", "american"
    ]
    
    polite_stopwords = [
        "dm", "please", "thanks", "thank", "sent", "send", "follow", "following","amp"
    ]

    print("-> Phase 1: Clustering and Raw Keywords...")
    vectorizer_raw = CountVectorizer(stop_words=base_stopwords, ngram_range=(1, 2))
    
    topic_model = BERTopic(
        language="english",
        vectorizer_model=vectorizer_raw,
        min_topic_size=15,  
        nr_topics=13,         
        calculate_probabilities=False,
    )

    # Compute the topics
    topics, probabilities = topic_model.fit_transform(
        tweets_texts,
        embeddings=precalculated_embeddings
    )

    topics_dict = {}
    topic_info = topic_model.get_topic_info()

    for index, row in topic_info.iterrows():
        topic_id = row['Topic']

        
        if topic_id == -1:
            continue

        
        words_raw = [{"word": w[0], "weight": float(w[1])} for w in topic_model.get_topic(topic_id)]
        
        topics_dict[topic_id] = {
            "id": int(topic_id),
            "count": int(row['Count']),
            "words_raw": words_raw
        }
    print("-> Phase 2: Updating with No-Airlines...")
    vectorizer_nobrands = CountVectorizer(stop_words=base_stopwords + airline_stopwords, ngram_range=(1, 2))
    
    
    topic_model.update_topics(tweets_texts, vectorizer_model=vectorizer_nobrands)
    
    for topic_id in topics_dict.keys():
        words_nb = [{"word": w[0], "weight": float(w[1])} for w in topic_model.get_topic(topic_id)]
        topics_dict[topic_id]["words_no_brands"] = words_nb

    
    print("-> Phase 3: Updating with No Airline and no courtesy words...")
    vectorizer_operational = CountVectorizer(stop_words=base_stopwords + airline_stopwords + polite_stopwords, ngram_range=(1, 2))
    
    topic_model.update_topics(tweets_texts, vectorizer_model=vectorizer_operational)
    
    for topic_id in topics_dict.keys():
        words_op = [{"word": w[0], "weight": float(w[1])} for w in topic_model.get_topic(topic_id)]
        topics_dict[topic_id]["words_operational"] = words_op

   
   
    global_topics_data = list(topics_dict.values())

    outlier_count = topics.count(-1)
    outlier_percentage = (outlier_count / len(topics)) * 100

    print(f"\nTotal Tweets: {len(topics)}")
    print(f"Outliers (Topic -1): {outlier_count} tweets ({outlier_percentage:.2f}%)")
    print(f"BERTopic is ready! Found {len(global_topics_data)} valid topics with 3 lenses.")

    try:
        with open("bertopic_data.json", "w", encoding="utf-8") as f:
            json.dump(global_topics_data, f, ensure_ascii=False, indent=4)
        print("Topic data successfully saved to 'bertopic_data.json'")
    except Exception as e:
        print(f"Error saving JSON file: {e}")
    
    
    return topics

## Visual analytics features

In [14]:
global_umap_mapper = None
global_topics_data = []

# 1. Scatterplot using UMAP
def generate_base_map_umap():
    global global_umap_mapper
    global global_topics_data

    
    csv_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "base_map.csv")
    model_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR,"umap_mapper.joblib")
    topics_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "bertopic_data.json")
    embeddings_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "embeddings_1024d.pt")

    tweets_dataset_path = os.path.join(BASE_INPUT_PATH, TWEETS_DIR, "tweets.csv")
    
    
    if os.path.exists(csv_path) and os.path.exists(model_path) and os.path.exists(topics_path):
        print(f"Found existing '{csv_path}' and '{model_path}'.")
        print("Loading UMAP model into memory... (Skipping heavy embedding computation!)")
        global_umap_mapper = joblib.load(model_path)

        df_saved = pd.read_csv(csv_path)
        setup_tfidf(df_saved['text'].astype(str).tolist())
        
        
        with open(topics_path, 'r') as f:
            global_topics_data = json.load(f)
        print("Loaded BERTopic data from JSON.")
        
        print("System ready.")
        return

    print("Files not found. Loading dataset to generate base map from scratch...")
    
    df_full = pd.read_csv(tweets_dataset_path, dtype={"tweet_id": str})
    df_sample = df_full.copy()

    # TF-IDF
    texts = df_sample['text'].astype(str).tolist()
    setup_tfidf(texts)
    
    texts = df_sample['text'].astype(str).tolist()
    
    
        

    if os.path.exists(embeddings_path):
        print("Saved embeddings found! Loading into RAM...")
        embeddings_1024d = torch.load(embeddings_path)
    else:
        print("No embeddings found. Computing from scratch with the Qwen model (this may take some time)...")
        embeddings_1024d = batch_get_embeddings(texts, batch_size=32)
    
    
        embeddings_save_path = os.path.join(BASE_WORKING_PATH, "embeddings_1024d.pt")
        torch.save(embeddings_1024d, embeddings_save_path)
        print(f" Embeddings successfully saved to: {embeddings_save_path}")
        # BERTopic
        print("\nExtracting Topics with BERTopic...")
    
    document_topics = initialize_bertopic(texts, embeddings_1024d.numpy()) 
    
    
    print("\nTraining UMAP for 2D projection...")
    
   
    umap_model = umap.UMAP(
        n_neighbors=15, 
        min_dist=0.1, 
        metric='cosine', 
        random_state=42
    )
    
    
    embeddings_2d = umap_model.fit_transform(embeddings_1024d.numpy())
    
    
    global_umap_mapper = umap_model
    
    df_map = pd.DataFrame({
        "tweet_id": df_sample["tweet_id"],
        "text": df_sample["text"],
        "sentiment": df_sample["airline_sentiment"],
        "airline": df_sample["airline"],
        "x": embeddings_2d[:, 0], 
        "y": embeddings_2d[:, 1], 
        "topic_id": document_topics 
    })
    csv_save_path = os.path.join(BASE_WORKING_PATH, "base_map.csv")
    model_save_path = os.path.join(BASE_WORKING_PATH, "umap_mapper.joblib")
    
    df_map.to_csv(csv_save_path, index=False)
    print("Base map generated successfully and saved as 'base_map.csv'.")
    
    
    joblib.dump(global_umap_mapper, model_save_path)



embedding_lock = threading.Lock()

def get_token_weights(tweet_text):
    with embedding_lock:
        orig_emb = get_embedding(tweet_text)

    
        doc = nlp(tweet_text)
        words = [token.text_with_ws for token in doc]

        results = []
        for i, word in enumerate(words):
            if not word.strip(): continue 
                
           
            mod_words = words.copy()
            mod_words.pop(i)
            mod_text = "".join(mod_words)
            
            if not mod_text.strip(): mod_text = "empty"
                
            
            mod_emb = get_embedding(mod_text)
            sim = F.cosine_similarity(orig_emb, mod_emb).item()
            impact = 1.0 - sim
            
            results.append({
                "word": word.strip(),
                "weight": round(impact, 4)
            })
        return results



def generate_heatmap_data(tweet_text):
    with embedding_lock:
        """Generates the flattened matrix data for the D3 Heatmap."""
        orig_emb = get_embedding(tweet_text)
        doc = nlp(tweet_text)
        
     
        valid_tokens = [t for t in doc if t.is_alpha]
        tokens_subset = valid_tokens[:10] 
        
        perturbations = ["Remove", "Dict_Synonym", "Dict_Antonym"]
        heatmap_data = []
        
        for token in tokens_subset:
            for pert in perturbations:
                
                mod_text = tweet_text 
                importance = 0.0
                
                if pert == "Remove":
                    
                    mod_text = "".join([t.text_with_ws if t.i != token.i else t.whitespace_ for t in doc])
                    
                elif pert == "Dict_Synonym":
                    replacement = get_single_word_replacement(token, mode="synonym")
                    if replacement:
                       
                        mod_text = "".join([t.text_with_ws if t.i != token.i else replacement + t.whitespace_ for t in doc])
                        
                elif pert == "Dict_Antonym":
                    replacement = get_single_word_replacement(token, mode="antonym")
                    if replacement:
                        mod_text = "".join([t.text_with_ws if t.i != token.i else replacement + t.whitespace_ for t in doc])
                
            
                if mod_text != tweet_text:
                    if not mod_text.strip(): 
                        mod_text = "empty"
                        
                    mod_emb = get_embedding(mod_text)
                    sim = F.cosine_similarity(orig_emb, mod_emb).item()
                    importance = 1.0 - sim
                
                heatmap_data.append({
                    "token": token.text,
                    "perturbation": pert,
                    "importance": round(importance, 4)
                })
                
        return heatmap_data

def get_perturbed_coordinates(tweet_text, perturbation_type="llm_paraphrase", target_word=None, replacement_word=None):
    """
    Applies one of the 5 specific perturbations, embeds, and projects to UMAP.
    Valid perturbation_types: 'dict_synonym', 'dict_antonym', 'noise', 'llm_paraphrase', 'llm_opposite'
    """
  
    if global_umap_mapper is None:
        raise ValueError("UMAP mapper not initialized. Run generate_base_map_umap() first.")

    display_text = None 

    if perturbation_type == "dict_synonym":
        
        new_text, display_text = perturb_dict(tweet_text, mode="synonym", target_word=target_word)
    elif perturbation_type == "dict_antonym":
        
        new_text, display_text = perturb_dict(tweet_text, mode="antonym", target_word=target_word)
    elif perturbation_type == "dict_remove":
        new_text, display_text = perturb_dict(tweet_text, mode="remove", target_word=target_word)
    elif perturbation_type == "custom":
        new_text, display_text = perturb_dict(tweet_text, mode="custom", target_word=target_word, replacement_word=replacement_word)
    
    elif perturbation_type == "noise":
        new_text, display_text = perturb_noise(tweet_text)
    elif perturbation_type == "llm_paraphrase":
        new_text = perturb_llm(tweet_text, mode="paraphrase")
    elif perturbation_type == "llm_opposite":
        new_text = perturb_llm(tweet_text, mode="opposite")
    elif perturbation_type == "typo":
        new_text, display_text = perturb_typo(tweet_text) 
    elif perturbation_type == "shuffle":
        new_text, display_text = perturb_shuffle(tweet_text)
    elif perturbation_type == "leetspeak":
        new_text, display_text = perturb_leetspeak(tweet_text) 
    else:
        new_text = tweet_text
        
    if display_text is None:
        display_text = new_text
    new_emb_tensor = get_embedding(new_text)
    new_emb_numpy = new_emb_tensor.cpu().numpy()
    

    new_2d_coords = global_umap_mapper.transform(new_emb_numpy)
    
    return {
        "original_text": tweet_text,
        "new_text": new_text,
        "display_text": display_text,
        "perturbation_applied": perturbation_type,
        "new_x": float(new_2d_coords[0][0]),
        "new_y": float(new_2d_coords[0][1])
    }

## Main logic

In [ ]:
import warnings
import os
import subprocess
warnings.filterwarnings('ignore')

if __name__ == "__main__":
    
    print("--- STEP 1: INITIALIZING SYSTEM ---")
    generate_base_map_umap()
    
    
    print("\n--- STEP 2: STARTING API SERVER ---")
    

    print("Installing required server packages...")
    subprocess.run(["pip", "install", "-q", "ngrok", "flask", "flask-cors"])
    
    from flask import Flask, request, jsonify
    from flask_cors import CORS
    import ngrok
    from kaggle_secrets import UserSecretsClient

    app = Flask(__name__)
    CORS(app)

    @app.route('/api/perturb', methods=['POST'])
    def perturb_endpoint():
        data = request.json
        tweet_text = data.get('text')
        perturb_type = data.get('type', 'llm_paraphrase') 
        target_word = data.get('target_word', None) 
        replacement_word = data.get('replacement_word', None)
        
        try:
            with embedding_lock:
                result = get_perturbed_coordinates(
                    tweet_text, 
                    perturb_type, 
                    target_word=target_word, 
                    replacement_word=replacement_word
                )
            return jsonify(result)
        except Exception as e:
            
            return jsonify({"error": str(e)}), 500

    @app.route('/api/tfidf', methods=['POST'])
    def tfidf_endpoint():
        data = request.json
        selected_texts = data.get('texts', [])
        top_n = data.get('top_n', 10) # Read the limit chosen by the user, default 10
        
        if not selected_texts or global_vectorizer is None:
            return jsonify([])
    
        
        subset_matrix = global_vectorizer.transform(selected_texts)
        sum_scores = subset_matrix.sum(axis=0)
        scores_array = np.squeeze(np.asarray(sum_scores))
        
       
        top_indices = scores_array.argsort()[-top_n:][::-1]
        
        
        results = []
        for i in top_indices:
            if scores_array[i] > 0: 
                results.append({
                    "word": global_feature_names[i],
                    "score": float(scores_array[i])
                })
                
        return jsonify(results)

    @app.route('/api/token_weights', methods=['POST'])
    def token_weights_endpoint():
        data = request.json
        tweet_text = data.get('text')
        try:
            result = get_token_weights(tweet_text)
            return jsonify(result)
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/heatmap', methods=['POST'])
    def heatmap_endpoint():
        data = request.json
        tweet_text = data.get('text')
        try:
            result = generate_heatmap_data(tweet_text)
            return jsonify(result)
        except Exception as e:
            return jsonify({"error": str(e)}), 500

    @app.route('/api/topics', methods=['GET'])
    def get_topics():
        
        return jsonify(global_topics_data)

    
    print("Configuring ngrok...")
    try:
        user_secrets = UserSecretsClient()
        my_secret_token = user_secrets.get_secret("NGROK_TOKEN")
        
        listener = await ngrok.forward(PORT, authtoken=my_secret_token)
        public_url = listener.url()
        
        print(f"\n YOUR API IS LIVE AT: {public_url}")
        print("Copy this URL into your app.js fetch() calls!")
        
       
        app.run(port=PORT, threaded=False)

    except KeyboardInterrupt:
     
        print("\n Interruzione rilevata (Ctrl+C). Chiusura del tunnel in corso...")
        listener.close() 
        print(" Tunnel Ngrok chiuso e server spento in modo pulito.")
        
    except Exception as e:
        print("\n Errore imprevisto:")
        raise e

--- STEP 1: INITIALIZING SYSTEM ---
Found existing '/kaggle/input/datasets/claudiacornacchia/embedding/base_map.csv' and '/kaggle/input/datasets/claudiacornacchia/embedding/umap_mapper.joblib'.
Loading UMAP model into memory... (Skipping heavy embedding computation!)
Addestramento TF-IDF globale in corso...
TF-IDF pronto.
Loaded BERTopic data from JSON.
System ready.

--- STEP 2: STARTING API SERVER ---
Installing required server packages...
Configuring ngrok...

 YOUR API IS LIVE AT: https://transdesert-overhugely-alessandra.ngrok-free.dev
Copy this URL into your app.js fetch() calls!
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [23/Jul/2026 15:01:20] "OPTIONS /api/tfidf HTTP/1.1" 200 -
127.0.0.1 - - [23/Jul/2026 15:01:20] "POST /api/tfidf HTTP/1.1" 200 -
